# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Felipe Muñoz
- Nombre de alumno 2: Antonia Landaeta

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [1]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.12.13 environment at: C:\MDS_proyectos\MDS7202-Labs\.venv
Resolved 141 packages in 1.50s
 Downloaded libcst
 Downloaded grpcio
 Downloaded apache-airflow-core
 Downloaded polars-runtime-32
Prepared 74 packages in 5.46s
Uninstalled 1 package in 104ms
Installed 74 packages in 627ms
 + a2wsgi==1.10.10
 + aiosmtplib==5.1.1
 + aiosqlite==0.21.0
 + apache-airflow==3.2.2
 + apache-airflow-core==3.2.2
 + apache-airflow-providers-common-compat==1.15.0
 + apache-airflow-providers-common-io==1.7.3
 + apache-airflow-providers-common-sql==2.0.1
 + apache-airflow-providers-smtp==3.0.1
 + apache-airflow-providers-standard==1.14.0
 + apache-airflow-task-sdk==1.2.2
 + argcomplete==3.6.3
 + asgiref==3.11.1
 + cadwyn==7.1.0
 + cron-descriptor==2.1.0
 + croniter==6.2.2
 + deprecated==1.3.1
 + dill==0.4.1
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.136.3
 + fastapi==0.137.2
 + fastapi-cli==0.0.27
 + fsspec==2026.6.0
 + googleapis-common-protos==1.75.0
 + greenback==1.3.0
 + g

In [ ]:
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import numpy as np
import pandas as pd
import polars as pl
import numba
import plotly.express as px
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [4]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [11]:
# Escribe aquí tu código
df = load_all_serial(data_dir=DATA_DIR, n_batches=20)
df.dtypes

id                         object
name                       object
album_name                 object
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                     object
year                        int64
genre                      object
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object

In [7]:
df.memory_usage(deep=True)

Index                           132
id                         14200000
name                       13638614
album_name                 13876175
artists                    24000000
danceability                1600000
energy                      1600000
key                         1600000
loudness                    1600000
mode                        1600000
speechiness                 1600000
acousticness                1600000
instrumentalness            1600000
liveness                    1600000
valence                     1600000
tempo                       1600000
duration_ms                 1600000
lyrics                    284216252
year                        1600000
genre                      10839938
popularity                  1600000
total_artist_followers      1600000
avg_artist_popularity       1600000
artist_ids                 24000000
niche_genres               24000000
dtype: int64

In [10]:
# Definimos df_opt aplicando las conversiones
df_opt = df.copy().astype({
    "danceability": "float32",
    "energy": "float32",
    "loudness": "float32",
    "speechiness": "float32",
    "acousticness": "float32",
    "instrumentalness": "float32",
    "liveness": "float32",
    "valence": "float32",
    "tempo": "float32",
    "avg_artist_popularity": "float32",
    "key": "int16",
    "mode": "int16",
    "year": "int32",
    "popularity": "int32",
    "duration_ms": "int32",
    "total_artist_followers": "int32"
})

# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()


![uso de memoria](p1_1_uso_memoria.png)

### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

**Respuesta**
1. **Parquet** es un formato de código abierto orientado en organizar los archivos en columnas, en lugar de filas como lo hace CSV. La ventaja que tiene sobre este último para los datos analíticos es que al almacenar los datos por columna, los comprime más que CSV al tener guardar datos del mismo tipo, permitiendo lecturas más rápidas. En concreto, Parquet sigue la técnica *columnar storage* que es una forma de almacenamiento de datos que los ordena por columnas en *contiguous memory locations*. Esta forma de almacenamiento permite obtener consultas de columnas más rápidas al no tener que cargar *todas* las columnas por cada fila de memoria como lo hace *row storage*.
2. **Apache Arrow** es una infraestructura de software para el desarrollo de aplicaciones de análisis de datos que procesa *columnar data*. Parquet es una componente de Apache Arrow para el formateo de bases de datos en formato columnar, mientras que la libería Pandas permite la lectura y escritura de archivos Parquet mediante ``pyarrow``.
3. `float32` se ocupa cuando se quiere mayor velocidad de cómputo para hacer operaciones por sobre máxim precisión, pues solo ocupa la mitad de los bits que necesita `float64`. Los contextos donde la pérdida de precisión es irrelavante son desarrollo de videojuegos, procesamiento de audio y entrenamiento de redes neuronales.
4. Cuando se quiere estimar valores muy precisos, como el modelamiento de materiales para la construcción o posición en el GPS; o cuando se maneja escalas de magnitud extremas ($10^9$ no es lo mismo que $10^{12}$). Los riesgos que existen al reducir la precisión van de tener simulaciones inestables a edificios con peligro de derrumbre.
5. ``Polars`` que está orientado para bases de datos masivas y utiliza **Apache Arrow** y ``Dask`` cuando los datos superan la RAM disponible.
6. Se redujo en 12.9 MiB o en un 3.1%. Pensabamos que con los cambios tendría una reducción considerable, esto se debe a que gran parte de los datos 
7. Si intentaramos reducir `valence` de ``float32`` a ``float16``, tendremos una gran pérdida de precisión a cambio de mayor velocidad de cómputo. Si ocuparamos este cambio para el modelo entrenado en la sección 2, se corre es riesgo de que los modelos no terminen convergiendo a un resultado dada que la matemática detrás de ellas es afectada por menor precisión.

In [13]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

Aquí se tiene que aplicar el ThreadPoolExecutor en la lectura de los archivos, tal que en lugar de hacerlo secuencial, se harán todas a la vez.

In [ ]:
# Escribe aquí tu código
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en paralelo y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    paths_str = [str(p) for p in paths] # para la lectura de los parquet en el pd.read_parquet
    with ThreadPoolExecutor() as executor: # sin argumento se calcula automaticamente el optimo de hilos según mi pc
        dataframes = list(executor.map(load_batch, paths_str)) # con map aseguramos guardar el orden de los archivos
    return pd.concat(dataframes, ignore_index=True)

In [ ]:
# medicion con %timeit de ambas lecturas para cada batch
measurements = []

# Evaluamos desde 1 hasta 20 archivos
for n in range(1, 21):
    res_serial = %timeit -o -q load_all_serial(DATA_DIR, n_batches=n) # -o para almacenar resultado, -q para no printear el resuldado
    res_parallel = %timeit -o -q load_all_parallel(DATA_DIR, n_batches=n)

    # .average nos da el tiempo promedio en segundos de las ejecuciones
    measurements.append({
        "n_files": n,
        "Tiempo_Serial (s)": res_serial.average,
        "Tiempo_Paralelo (s)": res_parallel.average,
        "Speedup": res_serial.average / res_parallel.average
    })

# Creamos la data comparativa estructurada
df_comparativo = pd.DataFrame(measurements)

In [28]:
df_comparativo

,n_files,Tiempo_Serial (s),Tiempo_Paralelo (s),Speedup
0,1,0.066358,0.065760,1.009095
1,2,0.136463,0.179549,0.760036
2,3,0.337001,0.266515,1.264473
3,4,0.452844,0.332666,1.361258
4,5,0.599353,0.418011,1.433822
5,6,0.799943,0.375912,2.128003
6,7,0.433144,0.431860,1.002972
7,8,1.009454,0.594365,1.698374
8,9,1.036777,0.681289,1.521788
9,10,1.095924,0.778504,1.407730


**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [15]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

![tiempo de lectura](p1_2_tiempo_lectura.png)

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

**Respuesta**
1. Que sea **I/O-bound** significa que el proceso es limitado por la velocidad de inputs o outputs de operaciones, a diferencia de **CPU-bound** que es limitado por el uso del CPU. La lectura de archivos desde el disco es **I/O-bound**.
2. **GIL** es un mecanismo que limita la ejecución de hilos tal que sólo sea de forma secuencial. Esto existe para proteger el uso de la memoria, dado que Python utiliza conteo de referencia para la administración de memoria, protege el conteo ante las modidicaciones que pueden hacer se si ejecutan más de un thread a la vez. Este mecanismo limita el paralelismo e introduce un potencial riesgo en el caso que un hilo se trabe y no deja que otro empiece/retome su ejecución.
3. Utilizamos Python porque es un lenguaje de programación simple, intuitivo y rápido de desarrollar, mientras delega las tareas pesadas de cómputo a librerías de alto rendimiento como NumPy, Arrow o PyTorch. Estas librerías al ser escritas en lenguajes compilados, tienen la capacidad de liberar el GIL por completo y de esta manera obtener mayor velocidad de procesamiento mediante la paralelización y el uso eficiente de la memoria en el núcleo de procesamiento.
4. Conviene utilizar `ThreadPoolExecutor` cuando la operación es I/O-bound, o `ProcessPoolExecutor` cuando la operación es CPU-bound.
5. Introduce costos de inicialización, consumo de memoria, cola de tareas y sincronización. Si los archivos fueran muy pequeños, la paralelización por `ThreadPoolExecutor` correrá más lento que la versión lineal debido al overhead producido al crear pool de threads, pues el costo de paralelizar es más elevado que hacerlo de forma lineal para este escenario.
6. Se observa mejora en la lectura paralela, a lo largo del eje X, se mantiene siempre debajo de la curva de la lectura serial. A partir de los 6 de archivos, la diferencia de tiempos de lectura entre serial y paralelo se hace más grande.
7. Esto puede deberse que todos los hilos están compitiendo para ser leído por el disco y por la RAM, haciendo que el GIL de Python obligue a los hilos a hacer filas y turnarse al registrar los DataFrames en memoria. 

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [16]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [18]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [19]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

![tiempo de prediccion](p2_1_tiempo_prediccion.png)

![Tiempo de predicción logarítmica](p2_2_tiempo_prediccion_logaritmica.png)

![Speedup vs python puro](p2_1_speed_relativo.png)

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

In [29]:
pd.__version__

'2.3.3'

**Respuestas**
1. La vectorización en NumPy se refiere a aplicar operaciones en arreglos completos sin tener que ocupar loops explícitos. Esto lo puede hacer dado que estas operaciones son optimizadas usado C, eliminando el overhead del intérprete de Python.
2. JIT es un método tal que compila las funciones que se vayan a utilizar solo la primera vez que se ejecutan. El decorador @numba.njit instruye a Numba a operar en modo `nopython`, es decir, indica que se compile los decoradores sin que se utilice CPython.
3. Numba es más lento al inicio porque al ocupar JIT, se ejecuta solo por una vez la compilación de todas las funciones.
4. Polars es un librería de alto rendimiento alternativa a Pandas. Sus principales características es que es rápido, tiene un motor de consulta vectorizado al utilizar Apache Arrow 
5. Pandas está escrito en Python y C, opera bajo un modelo de ejecución eager que procesa cada línea de código secuencialmente en un solo hilo y duplica frecuentemente los datos en memoria RAM mediante bloques de arreglos heterogéneos. Por el contrario, Polars está escrito en Rust, utiliza un motor de ejecución lazy capaz de optimizar y paralelizar las consultas al usar todos los núcleos del procesador, y adopta el estándar Apache Arrow, lo que permite no realizar copias innecesarias.
6. Pandas es más lento que Numpy debido a que usa la capa de Python para ver el control de la indexación, introduciendo overheard; realizar copias para mantener separados los labels de las filas y columnas; manejar datos de tipo heterogéneos; y comprobar la presencia de valores nulos.
7. SIMD es una instrucción que permitee procesar múltiples datos con una sola instrucción de al CPU, tales como las operaciones aritméticas y lógicas. Contribuye a la aceleración de NumPy y Polars gracias a la vectorización que permite paralelizar las operaciones.
8. Conviene ocupar Numba sobre Numpy cuando se quiere optimizar tareas matemáticas y con ciclos. Conviene ocupar Polars sobre Pandas en las operaciones numéricas cuando se trabaja con volúmenes de datos medianos a grandes, gracias a que es más rápido en ese escenario al tener la paralelización, las optimizaciones del motor lazy y el bajo consumo de memoria de Polars.
9. En nuestra medición, la implementación más rápida es Numba-JIT. Este resultado es esperable gracias al JIT elimina la sobrecarga de CPython, utilizando menos RAM y aprovechando la paralelización en la CPU.
10. Entre Pandas y Numpy, tienen una diferencia notable en sus tiempos de medición en las primeras 15K filas, con pandas siendo más rápido en este escenario. Esta diferencia se va achicando a medida que aumentan las miles de filas hasta que Numpy llega a superar en velocidad cerca de las 40K filas. Que NumPy sea lento en un principio se debe a la sobrecarga inicial por la inicialización de los arreglos en memoria, pero a medida que aumenta las filas el costo computacional de la administración interna de Pandas termina siendo mayor que el de NumPy, volviedose cada vez más lento que NumPy.
11. A partir de las primeras 500 filas aumenta la diferencia de velocidades entre Numba/NumPy vs Python puro.
12. A partir de las 12k filas en adelante, Polars es más eficiente que Pandas. La versión que tenemos de Pandas es la 2.3.3, como cae dentro de las versiones 2.x.x, se puede usar de forma opcional el backend PyArrow, permitiendo leer las tablas de forma columnar, ganando así más eficiencia en las operaciones. Sin embargo, Polars seguiría siendo más rapido por su motor lazy de evaluación que entrega un plan de ejecución optimizado antes de ejecutar, a diferencia de Pandas que usa eager; y por la paralelización.
13. Numba puede igualar o superar a NumPy en bucles numéricos simples porque compila la función completa a código de máquina para el procesador y evita la creación de arreglos intermedios en memoria, fusionando todo el cálculo dentro de un único bucle nativo, algo que NumPy no puede hacerlo al generar subarreglos para el almacenamiento de resultados intermedios.
14. Si incluyeramos el costo de conversión en la medición, el rendimiento de NumPy y Polars más bajo en los volúmenes bajos de datos. En el escenario de producción ese costo donde no existiría este costo, los datos tendrían que leerse directamente desde archivos Parquet para Polars. 
15. En ese escenario ocuparía Polars porque mi RAM sufriría sino lo ocupo gracias al motor lazy. En el caso que tenga una GPU, ocuparía JAX al ser la máz recomendada para la ciencia de datos.

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [21]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

n_jobs=1  → tiempo: 294.7s | RMSE: 0.1652
n_jobs=-1 → tiempo: 54.2s | RMSE: 0.1652


In [22]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

![tiempo de entrenamiento](p2_2_tiempo_entrenamiento_njobs.png)

![rmse](p2_2_rmse_njobs.png)

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

**Respuesta**
1. `n_jobs` representa la cantidad de threads que se deben usar para ejecutar tareas en paralelo. En el código previo, que sea 1 significa que se hace todo en serie mientras que -1 significa que se usa todos los procesadores.
2. Aquí el paralelismo real funciona porque la librería scikit-learn utiliza la librería joblib para acceder al paralelismo. Esta librería tiene la particularidad de usar un backend basado en el multiprocesamiento, él cual clona el programa en múltiples procesos de Python independientes y aislados, y como cada uno tiene su propia memoria y GIL, permite el uso de todos los procesadores a la vez.
3. Se redujo en un 81,6% al pasar a `n_jobs=-1` y obtubo un speedup 5.44 veces más rápido que solo usar una CPU.
4. En mi caso, tengo 4 núcleos, por lo que no es proporcional al speedup conseguido. Que haya superado al número de CPUs puede deberse al multiprocesamiento de Joblib a través de sus procesos aislados.
5. No hubo diferencia de RMSE entre ambas versiones. Este resultado es esperable porque la métrica de error es independiente al tiempo de procesamiento que toma la estimación.

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


In [ ]:
# Escribe aquí tu código (copia el template y completa los TODOs)


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>